# 08 · Resultados finales, comparación y conclusiones — Oslo Pollinator Lab

**Proyecto 2, UVG CC3074.** Este notebook integra evidencia existente, **no repite las 75 240 simulaciones de búsqueda NSGA-II** ni modifica archivos originales de los integrantes.

**Pregunta:** ¿cómo cambia el rendimiento de un simulador agregado de forrajeo al ubicar K=5 colmenas mediante RANDOM, GREEDY o NSGA-II, bajo un paisaje de parques reales de Oslo y supuestos ecológicos sintéticos?

**Distinción indispensable:** la geografía y las observaciones Dryad son reales; el recurso del motor se expresa en **synthetic_RU**, sin equivalencia probada con flores, polen, néctar ni polinización efectiva. La comparación empírica con Dryad utiliza **referentes estadísticos independientes**, no es validación directa de simulate().

**Contexto:** 737 parques GIS, 375 identificadores de candidatos (351 coordenadas únicas), horizonte de 18 días del 18-jul al 04-ago de 2017. El escenario histórico de los algoritmos usa motor 1.1.0; sensibilidad y demos React usan 1.1.1. Nunca mezclar ambas cohortes como un único experimento.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config/project2/base.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecutar desde la raíz del repositorio o desde notebooks/project2.")
OUT = ROOT / "results/project2/final_synthesis"
OUT.mkdir(parents=True, exist_ok=True)

def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Falta evidencia: {path}. Revisar README_REACT.md y scripts.")
    return pd.read_csv(path)

summary = read_csv("results/project2/integration/comparison_summary.csv")
paired = read_csv("results/project2/integration/comparison_paired.csv")
audit = read_csv("results/project2/integration/comparison_audit.csv")
changes = read_csv("results/project2/scientific_validation/archived_changes.csv")
front = read_csv("results/project2/integration/comparison_nsga_installations.csv")
history = read_csv("results/project2/nsga2/history.csv")
empirical = read_csv("results/project2/scientific_validation/empirical_errors.csv")
flowers_eval = read_csv("results/project2/scientific_validation/flowers_evaluation.csv")
activity_eval = read_csv("results/project2/scientific_validation/activity_evaluation.csv")
sensitivity = read_csv("results/project2/sensitivity_v111/summary.csv")
sens_reps = read_csv("results/project2/sensitivity_v111/replicates.csv")
meta = json.loads((ROOT / "results/project2/integration/comparison_metadata.json").read_text(encoding="utf-8"))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"Evidencia cargada desde {ROOT}")
print("Resumen histórico:", len(summary), "filas; sensibilidad:", len(sens_reps), "corridas.")

## 1. Identidad experimental y comprobaciones

Las comparaciones históricas usan **70 seeds comunes, réplicas 31–100**. RANDOM tiene 30 instalaciones fijas y se promedia primero por seed, **no** se trata a sus 2 100 filas como muestras independientes. GREEDY usa una instalación y NSGA-II una solución ilustrativa del frente, elegida con medias de la búsqueda. La cohorte reservada ya fue inspeccionada por el equipo: análisis **descriptivo**, no prueba confirmatoria prerregistrada.

La evidencia de sensibilidad es de motor 1.1.1 y se analiza **por separado**. Los intervalos bootstrap son condicionales a las ubicaciones guardadas y no incluyen incertidumbre de nuevas búsquedas del optimizador.

In [ ]:
assert set(audit["method"]) == {"RANDOM", "GREEDY", "NSGA2"}
assert audit["K"].eq(5).all()
assert audit["holdout_replications"].eq(70).all()
assert audit["engine_version"].nunique() == 1
assert audit["resource_unit"].eq("synthetic_RU").all()
assert summary.groupby("method")["metric"].nunique().eq(9).all()
assert summary["n_seeds"].eq(70).all()
assert len(sens_reps) == 1200
assert sens_reps["engine_version"].nunique() == 1
assert sens_reps["engine_version"].iloc[0] != audit["engine_version"].iloc[0]
assert sens_reps.groupby(["experiment_condition", "method"])["replication_id"].nunique().eq(30).all()
assert sens_reps["experiment_condition"].nunique() == 14
assert meta["candidate_ids"] == 375 and meta["unique_candidate_coordinates"] == 351

checks = pd.DataFrame([
    ("Histórico: tres métodos, K=5 y 70 seeds", "Cumple"),
    ("Nueve métricas por método, mismas unidades", "Cumple"),
    ("Histórico sintético: una sola versión del motor", "Cumple"),
    ("Sensibilidad: 1 200 corridas, 14 condiciones y 30 seeds", "Cumple"),
    ("Cohortes 1.1.0 y 1.1.1 separadas", "Cumple"),
    ("Validez ecológica directa del motor", "No demostrada"),
], columns=["Comprobación", "Estado"])
display(checks)
display(audit[["method", "original_rows", "solutions", "holdout_replications",
               "engine_version", "resource_unit"]])

## 2. ¿Qué mejoró en el escenario sintético?

Se reportan las **mismas unidades y las mismas condiciones**. Más synthetic_RU es favorable; menor distancia media de ida es favorable; otras métricas representan objetivos distintos o diagnósticos y no deben reducirse a un ranking universal.

**Advertencia:** la presión de competencia es cercana a cero en este experimento; una reducción relativa grande de un valor diminuto no demuestra que la competencia real de Oslo sea baja.

In [ ]:
selected_metrics = [
    "total_resource_obtained", "mean_distance_m", "p95_distance_m",
    "min_resource_per_hive", "competition_pressure", "hive_balance_jain",
    "patches_used_count"
]
metric_titles = {
    "total_resource_obtained": "Recurso obtenido (synthetic_RU)",
    "mean_distance_m": "Distancia media de ida (m)",
    "p95_distance_m": "P95 de distancia de ida (m)",
    "min_resource_per_hive": "Recurso mínimo por colmena (synthetic_RU)",
    "competition_pressure": "Presión de competencia (fracción)",
    "hive_balance_jain": "Equilibrio de Jain (índice)",
    "patches_used_count": "Parches utilizados (conteo)"
}
comparison_table = (summary.loc[summary.metric.isin(selected_metrics)]
                    .pivot(index="metric", columns="method", values="mean")
                    .reindex(selected_metrics)[["RANDOM", "GREEDY", "NSGA2"]])
comparison_table.index = [metric_titles[k] for k in selected_metrics]
display(comparison_table.round(4))
comparison_table.to_csv(OUT / "metrics_three_methods.csv", index_label="metric")

In [ ]:
def archived_bar(metric, label, output):
    order = ["RANDOM", "GREEDY", "NSGA2"]
    df = summary.loc[summary.metric.eq(metric)].set_index("method").loc[order]
    means = df["mean"].to_numpy(float)
    yerr = np.vstack([means - df["ci95_low"].to_numpy(float),
                      df["ci95_high"].to_numpy(float) - means])
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(["RANDOM", "GREEDY", "NSGA-II"], means, yerr=yerr, capsize=5,
           color=["#a4a79f", "#87aa93", "#d4ae69"])
    ax.set(ylabel=label, xlabel="Estrategia", title=f"Comparación histórica: {label}")
    ax.text(0.01, 0.98, "K=5 · 70 seeds · motor sintético 1.1.0 · IC 95% condicional",
            transform=ax.transAxes, va="top", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT / output, dpi=160)
    plt.show()

archived_bar("total_resource_obtained", "Recurso total entregado (synthetic_RU)",
             "comparison_resource.png")
archived_bar("mean_distance_m", "Distancia media de ida (m)",
             "comparison_distance.png")

## 3. Magnitud de la mejora y diferencias emparejadas

Para cada seed común se evalúa la diferencia entre métodos. La columna de **cambio porcentual** indica aumento (+) o disminución (−), **no** mejora automática: para distancia queremos reducir y para recurso queremos aumentar.

El IC de diferencias es puntual, condicionado a ubicaciones fijas y a la media de referencia usada como denominador para el porcentaje. No se deben interpretar estos intervalos como incertidumbre de nuevas instalaciones aleatorias ni de varias búsquedas NSGA-II.

In [ ]:
contrasts = ["GREEDY-RANDOM", "NSGA2-RANDOM", "NSGA2-GREEDY"]
focus = ["total_resource_obtained", "mean_distance_m", "p95_distance_m"]
paired_view = changes.loc[changes.contrast.isin(contrasts) & changes.metric.isin(focus),
                          ["contrast", "metric", "difference", "ci95_low", "ci95_high",
                           "percent_change", "n_pairs"]].copy()
paired_view["metric"] = paired_view["metric"].map(metric_titles)
paired_view["orientación"] = np.where(
    paired_view["metric"].str.contains("Recurso"), "aumentar", "disminuir"
)
display(paired_view.round(3))
paired_view.to_csv(OUT / "paired_comparisons.csv", index=False)

def historical_mean(method, metric):
    return float(summary.loc[(summary.method == method) & (summary.metric == metric),
                             "mean"].iloc[0])

nsga_vs_random_resource = 100 * (historical_mean("NSGA2","total_resource_obtained") /
                                 historical_mean("RANDOM","total_resource_obtained") - 1)
nsga_vs_random_distance = 100 * (1 - historical_mean("NSGA2","mean_distance_m") /
                                 historical_mean("RANDOM","mean_distance_m"))
print(f"NSGA-II vs RANDOM: +{nsga_vs_random_resource:.2f}% RU sintéticas; "
      f"-{nsga_vs_random_distance:.2f}% distancia media.")
print("No equivalen a cambios observados de polinización en Oslo.")

## 4. NSGA-II no tiene un ganador universal

El frente archivado contiene **49 IDs de solución, pero 32 configuraciones físicas diferentes**. La muestra ilustra el compromiso entre maximizar synthetic_RU y minimizar distancia. Una ubicación con más recurso puede exigir vuelos más largos.

La alternativa pareto_047 fue elegida de forma ilustrativa con resultados de búsqueda, no escogida para maximizar el conjunto evaluado. Un frente archivado no prueba convergencia al óptimo global; faltan varias semillas de optimizador. Los IDs candidatos distintos con coordenadas iguales también requieren una política futura de colocalización.

In [ ]:
unique_front = front.loc[~front["physical_duplicate"].fillna(False)].copy()
assert len(front) == 49 and len(unique_front) == 32
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(unique_front["mean_distance_m"], unique_front["total_resource_obtained"],
           alpha=.8, label="32 instalaciones físicas NSGA-II", color="#39806c")
for solution in ["pareto_047", "pareto_013", "pareto_045"]:
    item = unique_front.loc[unique_front.solution_id.eq(solution)]
    if item.empty:
        continue
    r = item.iloc[0]
    ax.scatter([r.mean_distance_m], [r.total_resource_obtained],
               color="#d6a54a", edgecolor="black", zorder=3)
    ax.annotate(solution, (r.mean_distance_m, r.total_resource_obtained),
                xytext=(8, 7), textcoords="offset points", fontsize=9)
for method, color in [("RANDOM", "#838b96"), ("GREEDY", "#7057a0")]:
    ax.scatter([historical_mean(method,"mean_distance_m")],
               [historical_mean(method,"total_resource_obtained")],
               color=color, marker="X", s=85, label=method, zorder=4)
ax.set(xlabel="Distancia media de ida (m)",
       ylabel="Recurso total (synthetic_RU)",
       title="Compromiso recurso–distancia: instalaciones archivadas")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(OUT / "pareto_tradeoff.png", dpi=160)
plt.show()

display(history[["generation", "front_size", "best_total_resource_obtained",
                 "best_mean_distance_m"]].tail(6))

## 5. Prueba de sensibilidad — 14 condiciones, 1 200 corridas

Las condiciones modifican **un factor por vez** (OAT), con 30 seeds comunes por método y condición disponible. En la sensibilidad se ejecuta motor 1.1.1, con GREEDY recalculado pero las ubicaciones NSGA-II de pareto_047 **congeladas**, sin reoptimización. No hay comparación NSGA-II para K=3/8. Un gráfico de barras permite contrastar escenarios sin sugerir que forman una serie temporal.

La pregunta no es únicamente quién gana en base, sino **si el orden cambia al modificar las hipótesis**.

In [ ]:
scenario_labels = {
    "base": "Base", "radius_short": "Radio 1 500 m",
    "radius_long": "Radio 5 000 m", "regeneration_off": "Sin regeneración",
    "activity_high": "Más oportunidades"
}
focus_sens = sensitivity.loc[
    sensitivity["condition"].isin(scenario_labels) &
    sensitivity["metric"].eq("total_resource_obtained")
].copy()
sens_table = (focus_sens.pivot(index="condition", columns="method", values="mean")
              .reindex(scenario_labels)[["RANDOM", "GREEDY", "NSGA2"]])
sens_table.index = [scenario_labels[k] for k in scenario_labels]
display(sens_table.round(2))
sens_table.to_csv(OUT / "sensitivity_selected.csv", index_label="condition")

fig, ax = plt.subplots(figsize=(11, 5.4))
sens_table.plot(kind="bar", ax=ax, width=.72,
                color=["#a4a79f", "#87aa93", "#d4ae69"])
ax.set(xlabel="Condición experimental (no es una secuencia temporal)",
       ylabel="Recurso obtenido (synthetic_RU)",
       title="Sensibilidad OAT, 30 seeds comunes por condición")
ax.tick_params(axis="x", rotation=18)
ax.legend(["RANDOM", "GREEDY", "NSGA-II congelado"])
fig.tight_layout()
fig.savefig(OUT / "sensitivity_resource.png", dpi=160)
plt.show()

print("ATENCIÓN: cambiar K también modifica el total de oportunidades de viaje; "
      "no es una comparación con esfuerzo total constante.")

## 6. Datos reales de Oslo: ¿qué se validó de verdad?

Dryad aporta **593 registros florales (100 traps)** y **1 297 registros de actividad (101 IDs)**. Tras reconstruir unidades con controles de calidad hay 99 unidades florales completas y 216 sesiones elegibles. Para referentes estadísticos independientes se separaron 79 traps para ajuste y 20 para evaluación en cada conjunto. Se calcularon MAE/RMSE e IC por bootstrap de sitios.

**Esto no calibra el simulador de colmenas.** Las RU no son flores/m², néctar ni polen; viajes/colmena/día no equivalen a registros/min de sesiones con detectabilidad desconocida. Los traps no tienen correspondencia geográfica confiable con los parques GIS. El clima observado es contexto, no una ley térmica incorporada al motor.

Una mejora del referente por estratos de temperatura para actividad es descriptiva para ese split; la estratificación mensual **empeoró** la predicción floral.

In [ ]:
empirical_view = empirical.loc[
    empirical["metric"].eq("MAE"),
    ["dataset", "method", "estimate", "ci95_low", "ci95_high", "unit",
     "train_sites", "test_sites"]
].sort_values(["dataset", "method"])
display(empirical_view.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4.8))
f = empirical_view.loc[empirical_view.dataset.eq("flowers")].set_index("method")
a = empirical_view.loc[empirical_view.dataset.eq("activity")].set_index("method")
# Dos unidades diferentes: dos figuras, jamás una escala compartida.
for dataset, df, unit, file in [
    ("Flores", f, "MAE (flores/m²)", "real_flowers_mae.png"),
    ("Actividad", a, "MAE (conteos registrados/min de sesión)", "real_activity_mae.png")
]:
    fig, ax = plt.subplots(figsize=(7, 4.1))
    df = df.loc[["global", "stratified"]]
    y = df["estimate"].to_numpy(float)
    err = np.vstack([y - df["ci95_low"].to_numpy(float),
                     df["ci95_high"].to_numpy(float) - y])
    ax.bar(["Media global", "Estratificada"], y, yerr=err,
           color=["#b8a380", "#699e90"], capsize=5)
    ax.set(ylabel=unit, title=f"Referentes Dryad: {dataset} (sitios de evaluación)")
    fig.tight_layout()
    fig.savefig(OUT / file, dpi=160)
    plt.show()

## 7. Tabla de correspondencia real / simulación base / optimización

La rúbrica pide las mismas métricas y unidades. Como las salidas del motor aún no tienen operador de observación compatible con Dryad, se declaran **ausencias en lugar de fabricar números**. La columna real para densidad y actividad resume las unidades de evaluación (no un dato de todo Oslo); no existe una medición real de synthetic_RU.

Esto documenta honestamente el **criterio parcialmente pendiente** de comparación ecológica. El estudio real sirve como referente y límite, pero **no prueba** el cambio ecológico causado por instalar las colmenas optimizadas.

In [ ]:
flowers_mean = float(flowers_eval["observed"].mean())
activity_mean = float(activity_eval["observed"].mean())
reality_table = pd.DataFrame([
    {"Métrica": "Densidad floral de transecto", "Unidad": "flores/m²",
     "Referencia real": round(flowers_mean, 3), "RANDOM (motor)": "No produce esta unidad",
     "NSGA-II (motor)": "No produce esta unidad"},
    {"Métrica": "Conteos de actividad por sesión", "Unidad": "conteos/min",
     "Referencia real": round(activity_mean, 3), "RANDOM (motor)": "No produce esta unidad",
     "NSGA-II (motor)": "No produce esta unidad"},
    {"Métrica": "Recurso entregado", "Unidad": "synthetic_RU",
     "Referencia real": "No disponible",
     "RANDOM (motor)": round(historical_mean("RANDOM", "total_resource_obtained"), 3),
     "NSGA-II (motor)": round(historical_mean("NSGA2", "total_resource_obtained"), 3)},
    {"Métrica": "Distancia media de ida", "Unidad": "m",
     "Referencia real": "No disponible",
     "RANDOM (motor)": round(historical_mean("RANDOM", "mean_distance_m"), 3),
     "NSGA-II (motor)": round(historical_mean("NSGA2", "mean_distance_m"), 3)}
])
display(reality_table)
reality_table.to_csv(OUT / "real_vs_simulated_scope.csv", index=False)

## 8. Interpretación científica final

**Qué mejoró en el escenario probado.** Con K=5 y 70 semillas compartidas, la distribución ilustrativa NSGA-II supera ampliamente a RANDOM en synthetic_RU y distancia media. GREEDY también mejora frente a RANDOM; las diferencias entre NSGA-II y GREEDY son menores. No todas las métricas favorecen el mismo método: GREEDY tiene un P95 de distancia menor que pareto_047 y RANDOM utiliza más parches.

**Bajo qué condiciones.** La sensibilidad indica que la posición relativa depende del radio y otros parámetros; con radio corto, GREEDY supera en recurso a las ubicaciones congeladas de NSGA-II. Reoptimizar podría cambiar el resultado, pero no se hizo en ese experimento. Competencia baja responde a las hipótesis sintéticas, no a datos ecológicos de Oslo.

**Qué tan confiable.** Comparaciones con seeds emparejadas, hashes, pruebas e IC condicionales son buenas evidencias internas. No demuestran que el motor represente cantidades reales recolectadas, que un cambio ocurriera en Oslo ni que la solución sea óptima globalmente. Falta cuantificar incertidumbre de varias búsquedas independientes y de nuevas ubicaciones aleatorias.

**Recomendación acotada.** En el escenario sintético de referencia, usar NSGA-II si interesa explorar compromisos entre objetivos y GREEDY si se requiere una solución rápida e interpretable. No recomendar instalar colmenas reales sin mediciones biológicas, permisos y validación externa.

**Alcance temporal:** 18 días, 18-jul a 04-ago 2017; **no** reconstrucción anual ni reproducción de toda la temporada.

In [ ]:
final_checks = pd.DataFrame([
    ("Problema real, datos y restricciones", "Respaldado con GIS y Dryad"),
    ("NSGA-II aplicado y comparado con RANDOM/GREEDY", "Respaldado, sintético"),
    ("70 seeds comunes + IC / error y 30 seeds sensibilidad", "Respaldado, con límites"),
    ("Validación ecológica directa del motor contra Oslo", "PENDIENTE"),
    ("Informe técnico PDF coherente con estas conclusiones", "PENDIENTE de preparación"),
    ("Diapositivas, contribuciones individuales y ensayo <=15 min", "PENDIENTE del equipo")
], columns=["Elemento de entrega", "Estado"])
display(final_checks)
final_checks.to_csv(OUT / "delivery_checks.csv", index=False)
print("Derivados disponibles en:", OUT.relative_to(ROOT))

## 9. Cómo reproducir y presentar

1. Instalar Python con requirements.txt; abrir este notebook desde la raíz o desde notebooks/project2 y ejecutar **Run All**. Las imágenes y tablas derivadas se guardarán en results/project2/final_synthesis/. No sobrescribe resultados históricos.
2. Ejecutar el frontend React/FastAPI desde **README_REACT.md** (no las instrucciones antiguas de Streamlit que figuraban en este notebook).
3. Demostración de 15 min: contexto/fuentes (2), mapa y modelo (2), tres algoritmos y Pareto (3), viajes/recursos (3), sensibilidad y evidencia real con límites (3), conclusiones/preguntas (2). Confirmar participación y aportes verificables.
4. Para el informe PDF usar **docs/project2/FINAL_SCIENTIFIC_SYNTHESIS.md** y completar el checklist de **docs/project2/FINAL_DELIVERY_CHECKLIST.md**.

**Nunca afirmar:** que el modelo predice polinización efectiva de Oslo, que RU equivale a néctar, que la validación de referentes empíricos valida simulate(), que 70 seeds son 2 100 unidades independientes o que el optimizador convergió al óptimo global.